# 04 — Model Analysis

Analyzes training results, per-dataset performance, confidence distribution, and error patterns.

| | |
|---|---|
| **Model** | best.pt (fine-tuned YOLOv8s) |
| **Dataset** | Merged from qasim21 + nexdata + roboflow |

In [ ]:
!pip install ultralytics pyyaml seaborn -q

import os, cv2, glob, random, shutil, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
from collections import Counter, defaultdict

sns.set_style('whitegrid')
print('Setup done!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/AI_TRAINING/GreenVision'
RUNS_DIR = os.path.join(DRIVE_ROOT, 'runs', 'person_detect_v1')
BEST_PT = os.path.join(RUNS_DIR, 'weights', 'best.pt')
DATA_YAML = '/content/data.yaml'
DATASET_DIR = '/content/dataset/merged'

## 1. Training Curves

In [ ]:
# Show training results (loss + metrics curves)
results_img = os.path.join(RUNS_DIR, 'results.png')
if os.path.exists(results_img):
    from IPython.display import Image as IPImage, display
    print('Training curves:')
    display(IPImage(filename=results_img, width=900))
else:
    print(f'results.png not found at {results_img}')
    print('Run 02_train.ipynb first.')

# Show confusion matrix if available
cm_img = os.path.join(RUNS_DIR, 'confusion_matrix.png')
if os.path.exists(cm_img):
    print('\nConfusion matrix:')
    display(IPImage(filename=cm_img, width=600))

## 2. Validation Metrics

In [ ]:
if not os.path.exists(BEST_PT):
    raise SystemExit(f'Model not found: {BEST_PT}. Run 02_train.ipynb first.')

model = YOLO(BEST_PT)
print(f'Loaded: {BEST_PT}')
print(f'Device: {model.device}')

if os.path.exists(DATA_YAML):
    metrics = model.val(data=DATA_YAML, device=0, verbose=False)

    print('=' * 45)
    print('  Validation Metrics')
    print('=' * 45)
    print(f'  mAP@0.5      : {metrics.box.map50:.4f}')
    print(f'  mAP@0.5:0.95 : {metrics.box.map:.4f}')
    print(f'  Precision    : {metrics.box.mp:.4f}')
    print(f'  Recall       : {metrics.box.mr:.4f}')
    f1 = 2 * metrics.box.mp * metrics.box.mr / (metrics.box.mp + metrics.box.mr + 1e-9)
    print(f'  F1 Score     : {f1:.4f}')
    print('=' * 45)
else:
    print('data.yaml not found. Skipping validation.')

## 3. Per-Dataset Performance

In [ ]:
# Run predictions on val set and group by source dataset
CONF = 0.25  # low conf for analysis, filter later
val_dir = os.path.join(DATASET_DIR, 'images', 'val')
val_lbl_dir = os.path.join(DATASET_DIR, 'labels', 'val')
val_imgs = glob.glob(os.path.join(val_dir, '*.jpg')) + glob.glob(os.path.join(val_dir, '*.png'))

print(f'Running inference on {len(val_imgs)} validation images...')

results = model.predict(val_imgs, conf=CONF, classes=[0], device=0, stream=True, verbose=False)

CONF_THRESH = 0.5
ds_stats = defaultdict(lambda: {'count': 0, 'detections': 0, 'confidences': [], 'gt_boxes': 0})

for r in results:
    fname = os.path.basename(r.path)
    # Parse dataset prefix: "qasim21_frame_00001.jpg" -> "qasim21"
    parts = os.path.splitext(fname)[0].split('_')
    ds_name = parts[0] if parts[0] in ['qasim21', 'nexdata', 'roboflow'] else 'unknown'

    n_det = len(r.boxes)
    confs = [float(b.conf[0]) for b in r.boxes] if n_det > 0 else []

    # Count ground truth boxes
    lbl_path = os.path.join(val_lbl_dir, os.path.splitext(fname)[0] + '.txt')
    n_gt = 0
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            n_gt = len([l for l in f.readlines() if l.strip()])

    ds_stats[ds_name]['count'] += 1
    ds_stats[ds_name]['detections'] += n_det
    ds_stats[ds_name]['confidences'].extend(confs)
    ds_stats[ds_name]['gt_boxes'] += n_gt

# Build summary table
rows = []
for ds, stats in sorted(ds_stats.items()):
    avg_det = stats['detections'] / stats['count'] if stats['count'] > 0 else 0
    avg_conf = np.mean(stats['confidences']) if stats['confidences'] else 0
    rows.append({
        'Dataset': ds,
        'Images': stats['count'],
        'GT Boxes': stats['gt_boxes'],
        'Pred Boxes': stats['detections'],
        'Avg Det/Img': f'{avg_det:.1f}',
        'Avg Conf': f'{avg_conf:.3f}',
    })

df = pd.DataFrame(rows)
print('\n')
print(df.to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

datasets = [r['Dataset'] for r in rows]
imgs = [r['Images'] for r in rows]
gt = [r['GT Boxes'] for r in rows]
pred = [r['Pred Boxes'] for r in rows]

axes[0].bar(datasets, imgs, color='#10b981')
axes[0].set_title('Images')
axes[0].bar_label(axes[0].containers[0])

axes[1].bar(datasets, gt, color='#3b82f6', label='Ground Truth')
axes[1].bar(datasets, pred, color='#f59e0b', alpha=0.7, label='Predicted')
axes[1].set_title('Boxes (GT vs Pred)')
axes[1].legend()

avg_confs = [float(r['Avg Conf']) for r in rows]
axes[2].bar(datasets, avg_confs, color='#8b5cf6')
axes[2].set_title('Avg Confidence')
axes[2].set_ylim(0, 1)
axes[2].bar_label(axes[2].containers[0], fmt='%.3f')

plt.suptitle('Per-Dataset Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Confidence Distribution

In [ ]:
# Collect all confidence scores
all_confs = []
for ds, stats in ds_stats.items():
    all_confs.extend(stats['confidences'])

if all_confs:
    plt.figure(figsize=(10, 5))
    plt.hist(all_confs, bins=30, color='#10b981', edgecolor='white', alpha=0.8)
    plt.axvline(np.mean(all_confs), color='red', linestyle='--', label=f'Mean={np.mean(all_confs):.3f}')
    plt.axvline(np.median(all_confs), color='blue', linestyle='--', label=f'Median={np.median(all_confs):.3f}')
    plt.xlabel('Confidence')
    plt.ylabel('Count')
    plt.title('Prediction Confidence Distribution', fontsize=13, fontweight='bold')
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f'Confidence stats:')
    print(f'  Count:  {len(all_confs)}')
    print(f'  Mean:   {np.mean(all_confs):.4f}')
    print(f'  Median: {np.median(all_confs):.4f}')
    print(f'  Min:    {np.min(all_confs):.4f}')
    print(f'  Max:    {np.max(all_confs):.4f}')
    print(f'  Std:    {np.std(all_confs):.4f}')
else:
    print('No detections found.')

## 5. Error Analysis

Finds false negatives (missed people) and shows detection failures.

In [ ]:
def compute_iou(box1, box2):
    """Compute IoU between two YOLO format boxes [cx, cy, w, h]."""
    x1_min = box1[0] - box1[2]/2
    y1_min = box1[1] - box1[3]/2
    x1_max = box1[0] + box1[2]/2
    y1_max = box1[1] + box1[3]/2

    x2_min = box2[0] - box2[2]/2
    y2_min = box2[1] - box2[3]/2
    x2_max = box2[0] + box2[2]/2
    y2_max = box2[1] + box2[3]/2

    xi_min = max(x1_min, x2_min)
    yi_min = max(y1_min, y2_min)
    xi_max = min(x1_max, x2_max)
    yi_max = min(y1_max, y2_max)

    inter = max(0, xi_max - xi_min) * max(0, yi_max - yi_min)
    area1 = box1[2] * box1[3]
    area2 = box2[2] * box2[3]
    union = area1 + area2 - inter

    return inter / union if union > 0 else 0

IOU_THRESH = 0.5
CONF = 0.25

false_negatives = []  # GT boxes with no matching prediction
false_positives = []  # Pred boxes with no matching GT
correct = []

val_imgs_list = glob.glob(os.path.join(val_dir, '*.jpg')) + glob.glob(os.path.join(val_dir, '*.png'))
results = model.predict(val_imgs_list, conf=CONF, classes=[0], device=0, stream=True, verbose=False)

for r in results:
    fname = os.path.basename(r.path)
    img_w, img_h = r.orig_shape[1], r.orig_shape[0]

    # Ground truth boxes
    lbl_path = os.path.join(val_lbl_dir, os.path.splitext(fname)[0] + '.txt')
    gt_boxes = []
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    gt_boxes.append([float(x) for x in parts[1:5]])

    # Predicted boxes
    pred_boxes = []
    if r.boxes is not None and len(r.boxes) > 0:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx = ((x1+x2)/2) / img_w
            cy = ((y1+y2)/2) / img_h
            bw = (x2-x1) / img_w
            bh = (y2-y1) / img_h
            pred_boxes.append([cx, cy, bw, bh])

    # Match predictions to GT
    matched_gt = set()
    matched_pred = set()

    for pi, pb in enumerate(pred_boxes):
        best_iou = 0
        best_gi = -1
        for gi, gb in enumerate(gt_boxes):
            iou = compute_iou(pb, gb)
            if iou > best_iou:
                best_iou = iou
                best_gi = gi
        if best_iou >= IOU_THRESH:
            matched_gt.add(best_gi)
            matched_pred.add(pi)
            correct.append(r.path)
        else:
            false_positives.append(r.path)

    for gi in range(len(gt_boxes)):
        if gi not in matched_gt:
            false_negatives.append(r.path)

print(f'=== Error Analysis (IoU threshold = {IOU_THRESH}) ===')
print(f'  Total images:       {len(val_imgs_list)}')
print(f'  Correct detections: {len(correct)}')
print(f'  False negatives:    {len(false_negatives)} (missed people)')
print(f'  False positives:    {len(false_positives)} (wrong detections)')

In [ ]:
# Show false negative examples (missed detections)
if false_negatives:
    fn_unique = list(set(false_negatives))[:6]
    fig, axes = plt.subplots(1, min(6, len(fn_unique)), figsize=(5*min(6, len(fn_unique)), 5))
    if len(fn_unique) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, fn_unique):
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        base = os.path.splitext(os.path.basename(img_path))[0]
        lbl = os.path.join(val_lbl_dir, base + '.txt')
        # Draw GT boxes in red (missed)
        if os.path.exists(lbl):
            with open(lbl) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cx, cy, bw, bh = map(float, parts[1:5])
                        x1 = int((cx - bw/2)*w)
                        y1 = int((cy - bh/2)*h)
                        x2 = int((cx + bw/2)*w)
                        y2 = int((cy + bh/2)*h)
                        cv2.rectangle(img, (x1,y1), (x2,y2), (255,0,0), 2)
                        cv2.putText(img, 'MISSED', (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 2)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(os.path.basename(img_path)[:30], fontsize=8)
    plt.suptitle('False Negatives (red = missed GT boxes)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No false negatives found!')

## 6. Dataset Summary

In [ ]:
# Overall dataset stats
train_imgs = glob.glob(os.path.join(DATASET_DIR, 'images', 'train', '*.*'))
val_imgs_all = glob.glob(os.path.join(DATASET_DIR, 'images', 'val', '*.*'))
train_lbls = glob.glob(os.path.join(DATASET_DIR, 'labels', 'train', '*.txt'))
val_lbls = glob.glob(os.path.join(DATASET_DIR, 'labels', 'val', '*.txt'))

# Count boxes
train_boxes = 0
for lbl in train_lbls:
    with open(lbl) as f:
        train_boxes += len(f.readlines())
val_boxes = 0
for lbl in val_lbls:
    with open(lbl) as f:
        val_boxes += len(f.readlines())

# Augmentation breakdown
orig = sum(1 for p in train_imgs if '_flip' not in os.path.basename(p) and '_bc' not in os.path.basename(p))
flip = sum(1 for p in train_imgs if '_flip' in os.path.basename(p))
bc = sum(1 for p in train_imgs if '_bc' in os.path.basename(p))

print('=' * 50)
print('  Final Dataset Summary')
print('=' * 50)
print(f'  Train images:   {len(train_imgs):>6d}')
print(f'    Original:     {orig:>6d}')
print(f'    Flip augment:  {flip:>6d}')
print(f'    BC augment:    {bc:>6d}')
print(f'  Val images:     {len(val_imgs_all):>6d}')
print(f'  Total images:   {len(train_imgs)+len(val_imgs_all):>6d}')
print(f'  Train boxes:    {train_boxes:>6d}')
print(f'  Val boxes:      {val_boxes:>6d}')
print(f'  Total boxes:    {train_boxes+val_boxes:>6d}')
print('=' * 50)

# Pie chart of dataset sources
ds_counts = Counter()
for p in train_imgs + val_imgs_all:
    name = os.path.basename(p)
    prefix = name.split('_')[0]
    if prefix in ['qasim21', 'nexdata', 'roboflow']:
        ds_counts[prefix] += 1
    else:
        ds_counts['other'] += 1

if ds_counts:
    fig, ax = plt.subplots(figsize=(7, 7))
    labels = list(ds_counts.keys())
    sizes = list(ds_counts.values())
    colors = ['#10b981', '#3b82f6', '#f59e0b', '#8b5cf6'][:len(labels)]
    ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    ax.set_title('Images per Source Dataset', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## Done!

Analysis complete. Review the metrics and error patterns to decide if more training data or epochs are needed.